# 06 - Feature Extraction

Turn every segmented heartbeat into one row of interpretable numeric features, across a small pool of records. No EDA or ML yet - this notebook ends with a saved table where **one row = one heartbeat**.

The reusable logic now lives in `src/` (`preprocessing.py`, `peak_detection.py`, `feature_extraction.py`): the same functions built and validated in notebooks 02-05, moved out of the notebooks now that several records need the identical pipeline. Record 100 still gives 2272 peaks and TP/FP/FN = 2271/1/2 through `src/`, so the move changed no behaviour.

## Choosing the record pool

Record 100 alone has 34 abnormal beats out of 2270, and because we will split train/test by *record*, a single record cannot even be split. We need several records - but not all 48.

`scripts/survey_records.py` ran our detector on every record (output saved in `results/metrics/record_survey.txt`). The pool is the set of records that pass three data-driven rules:

1. **Lead consistency** - MLII lead present and no paced beats (paced beats are out of scope, see notebook 05). Morphology features depend on the lead, so we do not mix leads.
2. **Detector quality gate** - precision and recall both >= 0.98 against the annotations. On a record where our simplified detector misses a third of the beats, those misses would silently drop beats from the dataset - and they are not random beats.
3. **Enough abnormal beats** - at least 50, so the record contributes meaningfully to the minority class.

The survey could not load record 208 (PhysioNet returned a 502 server error), so it was never evaluated - it is *not* excluded on quality.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.feature_extraction import (FEATURE_COLUMNS, build_beat_table, dominant_deflection,
                                    process_record)
from src.peak_detection import detect_r_peaks
from src.preprocessing import bandpass_filter, get_ecg_lead, load_record

rows = []
for line in Path("../results/metrics/record_survey.txt").read_text().splitlines()[2:]:
    if "|" not in line or "skipped" in line:
        continue
    left, right = line.split("|")
    record, leads, normal, abnormal, paced = left.split()
    tp, fp, fn, precision, recall, f1, _ = right.split()
    rows.append((record, leads, int(normal), int(abnormal), float(precision), float(recall), float(f1)))

survey = pd.DataFrame(rows, columns=["record", "leads", "normal", "abnormal", "precision", "recall", "f1"])
passes_gate = (survey.precision >= 0.98) & (survey.recall >= 0.98)
POOL = survey[passes_gate & (survey.abnormal >= 50)].record.tolist()

print(f"{len(survey)} records were eligible (MLII present, no paced beats); {passes_gate.sum()} pass the detector gate")
print(f"POOL ({len(POOL)} records): {POOL}")
print()
print("Records with >= 50 abnormal beats that are excluded ONLY because of detector quality:")
survey[~passes_gate & (survey.abnormal >= 50)]

**Selection bias to keep in mind.** The pool is, by construction, records where our simplified detector works. Anything we report later describes "records this detector handles", not the whole database. The excluded records are mostly the fixed-global-threshold failure documented in notebook 03 at its worst (e.g. 210 with recall 0.003; 207 with 1282 false positives). That is a scope decision to state in the README, not to hide.

## The feature set (13 features, all interpretable)

Each beat is a 600 ms window (200 ms before / 400 ms after the R peak) of the *filtered* signal. Amplitudes are in mV.

| Feature | What it represents | Why it may help classification | How it is calculated |
|---|---|---|---|
| `rr_pre_s` | Time since the previous R peak | Ectopic beats arrive *early*. In record 100, atrial premature beats have 0.52-0.68 s before them (median 0.60) vs 0.73-0.85 s for normal beats (5th-95th percentile) | `(r_peak[i] - r_peak[i-1]) / fs` |
| `rr_post_s` | Time until the next R peak | After a premature *ventricular* beat the following interval is typically long (a pause) | `(r_peak[i+1] - r_peak[i]) / fs` |
| `rr_ratio` | Early-arrival + pause pattern in one number | ~1.0 in regular rhythm; well below 1 when a beat arrives early and is followed by a pause; less dependent on the patient's resting heart rate. How strongly this holds per beat type is measured below, not assumed | `rr_pre_s / rr_post_s` |
| `heart_rate_bpm` | Instantaneous heart rate | Rhythm-rate context (e.g. tachycardia) | `60 / rr_pre_s` |
| `amp_mean` | Average level over the window | After filtering it sits near 0; departures come from residual baseline shift or a large sustained deflection | `beat.mean()` |
| `amp_std` | Spread of amplitudes | Tall and/or wide waveforms have a larger spread than a narrow normal spike | `beat.std()` |
| `amp_min` | Most negative sample | Polarity: a normal upright beat has a small negative dip; an inverted PVC goes strongly negative | `beat.min()` |
| `amp_max` | Most positive sample | Size of the upright deflection | `beat.max()` |
| `energy_mv2s` | Signal energy of the window | Wide, large PVCs carry far more energy than a narrow normal QRS | `sum(beat**2) / fs` (integral of x^2 dt) |
| `r_amplitude_mv` | Filtered voltage exactly at the detected R location | Amplitude of the detected peak (small if the detector landed off the true peak) | `beat[pre_samples]` |
| `dominant_deflection_mv` | Signed height of the largest deflection within +-100 ms of R, relative to the beat's median | Captures polarity *and* size: positive for a normal upright QRS, negative for an inverted one | value at `argmax(abs(x - median))` inside the QRS region |
| `qrs_p2p_mv` | Peak-to-peak excursion inside the QRS region | Total size of the QRS including its Q/S deflections | `max - min` inside +-100 ms |
| `qrs_fwhm_ms` | Width of the dominant deflection at half its height | PVCs are wide and blunt; a normal QRS spike is narrow (17-22 ms in record 100) | count contiguous same-sign samples >= half the extremum, divided by `fs` |

**Honest caveats about these features**

- `qrs_fwhm_ms` is the width of the *main spike*, not the clinical QRS duration (that would need Q-onset and S-offset detection, which we deliberately do not attempt).
- `heart_rate_bpm` is an exact function of `rr_pre_s` - redundant. Tree models are unaffected; the correlation analysis in the EDA stage will show it, and we can drop it there. We keep it now because it is a natural feature to inspect.
- `rr_post_s` and `rr_ratio` use the *next* beat, so this is an offline analysis - a real-time system would not have them.
- RR features inherit detector errors: a missed beat doubles an interval, a false positive halves it.
- Amplitude features depend on patient and electrode gain, so they will vary between records - exactly what a record-level split will expose.

## Seeing two features on real beats

One normal beat (`N`) and one premature ventricular contraction (`V`) from record 119, with the dominant deflection and its half-height width marked. This is a visual check that `dominant_deflection` and `qrs_fwhm_ms` measure what we claim they measure.

In [ ]:
demo_record, demo_annotation = load_record("119")
demo_signal, _ = get_ecg_lead(demo_record)
demo_fs = demo_record.fs
demo_filtered = bandpass_filter(demo_signal, 0.5, 40, demo_fs, order=4)
demo_table, _ = build_beat_table("119", demo_filtered, detect_r_peaks(demo_filtered, demo_fs), demo_fs, demo_annotation)

pre_samples = int(round(0.2 * demo_fs))
post_samples = int(round(0.4 * demo_fs))
t_ms = (np.arange(pre_samples + post_samples) - pre_samples) / demo_fs * 1000

examples = {
    "Normal beat (N)": demo_table[demo_table.symbol == "N"].iloc[50],
    "Premature ventricular contraction (V)": demo_table[demo_table.symbol == "V"].iloc[50],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, (title, row) in zip(axes, examples.items()):
    p = int(row.r_peak_sample)
    beat = demo_filtered[p - pre_samples:p + post_samples]
    d = dominant_deflection(beat, demo_fs, pre_samples)

    ax.plot(t_ms, beat, color="black", linewidth=1.2)
    ax.axhline(d["baseline"], color="grey", linestyle=":", label="baseline (beat median)")
    ax.axvspan(-100, 100, color="tab:blue", alpha=0.08, label="QRS region (+-100 ms)")
    ax.scatter(t_ms[d["peak_idx"]], beat[d["peak_idx"]], color="red", zorder=3, label="dominant deflection")
    ax.hlines(d["baseline"] + 0.5 * d["height"], t_ms[d["left_idx"]], t_ms[d["right_idx"]],
              color="tab:orange", linewidth=4, label=f"FWHM = {row.qrs_fwhm_ms:.0f} ms")
    ax.set_title(f"{title}\nrr_pre={row.rr_pre_s:.2f}s  rr_post={row.rr_post_s:.2f}s  "
                 f"deflection={row.dominant_deflection_mv:+.2f} mV")
    ax.set_xlabel("Time relative to detected R peak (ms)")
    ax.legend(loc="upper right", fontsize=8)
axes[0].set_ylabel("Amplitude (mV)")
fig.tight_layout()
fig.savefig("../results/figures/11_feature_examples.png", dpi=120)
plt.show()

pd.DataFrame(examples).loc[FEATURE_COLUMNS].astype(float).round(3)

## Building the table for the whole pool

`process_record` runs the full pipeline for one record: load -> bandpass filter -> detect R peaks -> segment -> label -> features. Records are downloaded once into `data/mitdb/` (gitignored) and read from disk afterwards.

**How labels are attached:** each detected peak is matched to the nearest annotated *beat* within 150 ms (non-beat markers such as `+` and `~` never label a beat). 150 ms is half the detector's 300 ms minimum peak spacing, so a detection can never be ambiguous between two annotated beats. The distance is kept in `ann_offset_ms`: a beat whose peak was mislocated (like the PVC in record 100, 61 ms off) stays visible in the table for error analysis instead of being silently dropped.

**What gets dropped, and is counted:** peaks whose window leaves the signal, the first/last beat of a record (no neighbour for the RR features), peaks with no annotation nearby (false positives - no ground truth to label them with), and paced beats.

In [ ]:
tables, summaries = [], {}
for name in POOL:
    table, summary = process_record(name)
    tables.append(table)
    summaries[name] = summary

beat_table = pd.concat(tables, ignore_index=True)

unrecognized = {n: s["unrecognized_symbols"] for n, s in summaries.items() if s["unrecognized_symbols"]}
print("Unrecognized annotation symbols:", unrecognized if unrecognized else "none")

summary_df = pd.DataFrame(summaries).T.drop(columns="unrecognized_symbols").fillna(0).astype(int)
per_record = beat_table.groupby("record").agg(normal=("is_abnormal", lambda s: int((s == 0).sum())),
                                              abnormal=("is_abnormal", "sum"))
per_record["abnormal_pct"] = (100 * per_record.abnormal / (per_record.normal + per_record.abnormal)).round(1)
summary_df.join(per_record)

## Sanity checks

In [ ]:
assert not beat_table[FEATURE_COLUMNS].isna().any().any(), "features must not contain NaN"
assert (beat_table.groupby("record").size() == summary_df.n_rows).all()

print("Rows (one per heartbeat):", len(beat_table))
print(beat_table.label.value_counts().to_string())
print(f"Abnormal fraction: {100 * beat_table.is_abnormal.mean():.1f}%")
top = per_record.abnormal.sort_values(ascending=False).head(3)
print("Largest abnormal-beat contributors:",
      ", ".join(f"{r} ({n}, {100 * n / per_record.abnormal.sum():.0f}% of all abnormal)" for r, n in top.items()))
print()
print("Matched peaks more than 50 ms from their annotation (mislocated), by label:")
print(beat_table.assign(mislocated=beat_table.ann_offset_ms > 50).groupby("label").mislocated.agg(["sum", "mean"]).round(4))
print()
print("Abnormal symbols in the pool:")
print(beat_table[beat_table.is_abnormal == 1].symbol.value_counts().to_string())

**Two things to carry into the next stages.** A single record (232) supplies a large share of all abnormal beats, so *which side of the train/test split it lands on* will matter a great deal - the record-level split needs to account for that. And the mislocated matches (> 50 ms from the annotation) are almost all abnormal: the polarity-blind refinement weakness from notebook 03 does not hurt normal and abnormal beats equally. It is under 1% of abnormal beats, but it is a systematic bias, not noise.

## An early read on the features, by beat type

A preview only - the real analysis is the EDA stage. Median feature values per annotation symbol (`N`, `L`, `R`, `j` are labeled Normal; `A`, `a`, `J`, `V`, `F` are labeled Abnormal), so the claims in the feature table are backed by numbers we can see.

In [ ]:
by_symbol = beat_table.groupby("symbol").agg(
    label=("label", "first"),
    beats=("symbol", "size"),
    rr_pre_s=("rr_pre_s", "median"),
    rr_ratio=("rr_ratio", "median"),
    qrs_fwhm_ms=("qrs_fwhm_ms", "median"),
    dominant_deflection_mv=("dominant_deflection_mv", "median"),
).round(3)
by_symbol.sort_values(["label", "beats"], ascending=[True, False])

**Reading it (not concluding from it):**

- `V` beats stand out on timing (`rr_ratio` about half that of `N`), on width (`qrs_fwhm_ms` more than twice as wide), and their median `dominant_deflection_mv` is negative - although individual PVCs can be upright (the record 119 example above), so the sign alone is not a rule.
- `A` beats look like normal beats in QRS width, and pool-wide their median `rr_ratio` is near 1. Their median `rr_pre_s` is not even shorter than `N`'s, even though within record 100 they clearly arrive earlier (0.60 s vs 0.80 s): an *absolute* interval is confounded by each patient's resting heart rate. The existing timing features therefore barely separate `A` from `N` across patients - expect atrial beats to be the hard part of the abnormal class. A *relative* prematurity measure (`rr_pre_s` divided by the recent average RR of the same record) would remove that confound; whether to add it is a decision for the EDA stage, where we can confirm `A` really is the problem instead of adding features pre-emptively.
- `L` and `R` beats are labeled *Normal* (bundle-branch-block variants) but are visibly wider than `N`, and `F` beats (labeled Abnormal) are similar in width - a plausible source of confusion later.

In [ ]:
r_amp = beat_table[beat_table.label == "Normal"].groupby("record").r_amplitude_mv.median().round(2)
print(f"Median R amplitude of Normal beats, per record: "
      f"lowest {r_amp.min()} mV (record {r_amp.idxmin()}), highest {r_amp.max()} mV (record {r_amp.idxmax()}) "
      f"- a {r_amp.max() / r_amp.min():.1f}x spread")

Absolute amplitude differs several-fold *between* records for the same beat type, so amplitude features (`amp_*`, `r_amplitude_mv`, `dominant_deflection_mv`, `qrs_p2p_mv`, `energy_mv2s`) partly encode "which patient/electrode" rather than "which beat type". Whether to normalise them per record is a decision for the EDA/ML stages (a per-record scale computed from that record's own signal, without labels, would not leak test information).

## Save the table

Written to `data/processed/beat_features.csv` (gitignored - it is regenerated in about a minute by running this notebook). Columns: `record`, `r_peak_sample`, `time_s`, `symbol`, `label`, `is_abnormal`, `ann_offset_ms`, then the 13 features. `record` is kept so the next stages can split by record, and `symbol` is kept (but is **not** a feature) so error analysis can say *which kind* of abnormal beat was missed.

In [ ]:
out_path = Path("../data/processed/beat_features.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
beat_table.to_csv(out_path, index=False)
print(f"Saved {len(beat_table)} rows x {beat_table.shape[1]} columns to {out_path}")
beat_table.head()